# Final true-graph reference baselines

This notebook computes the deterministic known-model reference policies used in
the final paper benchmark.

It intentionally performs **no graph learning**.

Two references are evaluated:

1. **`true_graph_static`** — the existing reference. It knows the exact
   influence matrix \(A\), computes the static graph centrality once, and uses
   the same centrality-plus-distance allocation heuristic as before. It is
   evaluated on **all 195 benchmark environments**.

2. **`true_graph_analytic`** — a stronger reference for dynamics whose exact
   propagation law admits a nonnegative control-relevant matrix in the same
   form used by the learned nonlinear controller. It is evaluated only for:

   - linear/Laplacian dynamics;
   - COCA;
   - nonlinear influence.

The analytic interaction matrices are

\[
M^\star(x)=A
\qquad\text{(linear)},
\]

\[
M^\star(x)=
\operatorname{diag}\!\left(x_i(1-x_i)\right)A
\qquad\text{(COCA)},
\]

and

\[
M^\star_{ij}(x)
=
A_{ij}
\frac{\tanh(\beta(x_j-x_i))}{x_j-x_i},
\]

with the continuous limit \(\alpha^\star_{ij}=\beta\) when \(x_j=x_i\),
for nonlinear influence.

The matrix is evaluated from the **current pre-intervention state**, exactly
where the learned controller evaluates its learned \(M_\phi(x)\). There is no
look-ahead or candidate-action simulation.

Hegselmann--Krause, Friedkin--Johnsen and repulsion retain only the static
true-graph reference:

- HK can produce disconnected active interaction graphs, so a unique global
  centrality is not generally defined.
- FJ contains an additive prejudice term that has no unique pairwise
  \(A\alpha(x)\) decomposition.
- Repulsion produces signed interaction weights, incompatible with the
  nonnegative centrality construction used by the controller.

This baseline run is shared by the abundant-data and single-shot learned
experiments because the benchmark environments, campaign horizon and control
budget are identical.


In [1]:
from __future__ import annotations

import hashlib
import inspect
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 300)


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "opinion_dynamics").exists():
            return candidate
    raise RuntimeError(
        "Could not find repository root containing opinion_dynamics/."
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from rl_envs_forge.envs.network_graph.network_graph import NetworkGraph
from rl_envs_forge.envs.network_graph.graph_utils import (
    compute_eigenvector_centrality,
    compute_laplacian,
)
from opinion_dynamics.baseline import centrality_based_continuous_control

try:
    GIT_COMMIT = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=REPO_ROOT,
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
except Exception:
    GIT_COMMIT = "unknown"

print("Repository root:", REPO_ROOT)
print("Git commit:", GIT_COMMIT)
print("NetworkGraph:", inspect.getsourcefile(NetworkGraph))
print(
    "centrality_based_continuous_control:",
    inspect.getsourcefile(centrality_based_continuous_control),
)


C:\Users\Chainsword\AppData\Local\Temp\ipykernel_79868\2684917984.py:14: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Repository root: d:\Work\repos\RL\unknown_graph_networks
Git commit: 1c1fdc6
NetworkGraph: c:\Users\Chainsword\anaconda3\envs\phd_rl_algos\Lib\site-packages\rl_envs_forge\envs\network_graph\network_graph.py
centrality_based_continuous_control: d:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\baseline.py


## Frozen benchmark configuration

In [2]:
STUDY_NAME = "true_graph_static_and_analytic_baselines_2026_09_28"
PIPELINE_VERSION = "2026-09-28-true-model-baselines-v1"

N = 15
TARGET = 1.0

NUM_CAMPAIGNS = 20
T_CAMPAIGN = 0.5
T_S = 0.1

MAX_U = 0.20
TOTAL_CONTROLLED_BUDGET = 6.0
B_CAMPAIGN = TOTAL_CONTROLLED_BUDGET / (NUM_CAMPAIGNS - 1)

UNSTRUCTURED_TOPOLOGY_SEEDS = [3, 4, 5, 6, 7]
UNSTRUCTURED_OPINION_SEEDS = [0, 1, 2, 4, 5]

UNSTRUCTURED_DYNAMICS = {
    "laplacian": {},
    "coca": {},
    "hegselmannkrause": {
        "hk_epsilon": 0.50,
        "hk_include_self": True,
    },
    "friedkinjohnsen": {
        "fj_lambda": 0.98,
    },
    "nonlinearinfluence": {
        "nonlinear_beta": 4.0,
    },
    "repulsion": {
        "repulsion_epsilon": 0.30,
        "repulsion_strength": 0.10,
    },
}

STRUCTURED_HK_EPSILON = 0.25
STRUCTURED_FJ_LAMBDA = 0.98
STRUCTURED_NLI_BETA = 4.0
STRUCTURED_ENVIRONMENT_SEEDS = [0, 1, 2, 3, 4]

HK_FAMILIES = {
    "ring_bridge": {
        "peer_pattern": "ring",
        "hub_leaf_weight": 3.0,
        "peer_weight": 0.40,
        "bridge_weight": 3.0,
    },
    "paired_bridge": {
        "peer_pattern": "paired",
        "hub_leaf_weight": 3.0,
        "peer_weight": 0.55,
        "bridge_weight": 3.0,
    },
    "dense_weak_bridge": {
        "peer_pattern": "dense",
        "hub_leaf_weight": 3.0,
        "peer_weight": 0.08,
        "bridge_weight": 3.0,
    },
}

BROADCAST_FAMILIES = {
    "asymmetric_broadcast": {
        "kind": "asymmetric",
        "hub_influence_total": 0.85,
    },
    "dual_broadcast": {
        "kind": "dual",
        "hub_influence_total": 0.85,
    },
    "split_broadcast": {
        "kind": "split",
        "hub_influence_total": 0.85,
    },
}

FJ_HUB_LEVEL = 0.16
FJ_LEAF_LEVEL = 0.48
NLI_HUB_LEVEL = 0.10
NLI_LEAF_LEVEL = 0.50

EDGE_WEIGHT_JITTER_FRAC = 0.05
HK_OPINION_JITTER = 0.006
BROADCAST_OPINION_JITTER = 0.010

ANALYTIC_DYNAMICS = {
    "laplacian",
    "coca",
    "nonlinearinfluence",
}

RESULTS_ROOT = (
    REPO_ROOT / "opinion_dynamics" / "experiments" / "results"
)
RESULTS_DIR = (
    RESULTS_ROOT
    / "experiment_2026_09_28_true_graph_static_and_analytic_baselines"
)
CACHE_ROOT = (
    RESULTS_ROOT
    / "_trial_cache"
    / STUDY_NAME
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print("Campaign budget:", B_CAMPAIGN)
print("Static baseline environments expected: 195")
print("Analytic baseline environments expected: 90")
print("Total deterministic rollouts expected: 285")


Campaign budget: 0.3157894736842105
Static baseline environments expected: 195
Analytic baseline environments expected: 90
Total deterministic rollouts expected: 285


## Exact benchmark graph helpers

In [3]:
def _normalize_rows(raw: np.ndarray) -> np.ndarray:
    raw = np.asarray(raw, dtype=float)
    np.fill_diagonal(raw, 0.0)
    sums = raw.sum(axis=1, keepdims=True)
    if np.any(sums <= 0):
        raise RuntimeError("Every graph row must have positive mass.")
    A = raw / sums
    np.fill_diagonal(A, 0.0)
    return A


def _jitter_positive_edges(
    raw: np.ndarray,
    rng: np.random.Generator,
    frac: float,
) -> np.ndarray:
    out = np.asarray(raw, dtype=float).copy()
    mask = out > 0
    multipliers = rng.uniform(
        1.0 - float(frac),
        1.0 + float(frac),
        size=out.shape,
    )
    out[mask] *= multipliers[mask]
    np.fill_diagonal(out, 0.0)
    return out


def _add_undirected(
    raw: np.ndarray,
    i: int,
    j: int,
    weight: float,
) -> None:
    raw[i, j] += float(weight)
    raw[j, i] += float(weight)


def centrality_from_A(A: np.ndarray) -> np.ndarray:
    v = compute_eigenvector_centrality(
        compute_laplacian(np.asarray(A, dtype=float))
    )
    v = np.asarray(v, dtype=float).reshape(-1)
    v = np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)

    if v.sum() < 0:
        v = -v

    v = np.maximum(v, 0.0)
    if v.sum() <= 1e-12:
        v = np.abs(v)

    if v.sum() <= 1e-12:
        return np.full(N, 1.0 / N)

    return v / v.sum()

## Exact unstructured benchmark generator

In [4]:
def build_unstructured_adjacency(topology_seed: int) -> np.ndarray:
    # Use the same graph-family settings as EnvironmentFactory's original
    # randomized environments, but extract A once and then reuse it for every
    # propagation law.
    x_dummy = np.linspace(0.0, 1.0, N)

    graph_env = NetworkGraph(
        num_agents=N,
        graph_model="barabasi_albert",
        ba_m=2,
        ba_prune_max_frac=0.5,
        ba_qsc_tol=1e-8,
        ba_max_tries=500,
        max_u=MAX_U,
        budget=1000.0,
        desired_opinion=TARGET,
        t_campaign=T_CAMPAIGN,
        t_s=T_S,
        max_steps=NUM_CAMPAIGNS + 20,
        opinion_end_tolerance=0.01,
        control_beta=0.4,
        normalize_reward=True,
        terminal_reward=0.0,
        seed=int(topology_seed),
        terminate_when_converged=False,
        dynamics_model="laplacian",
        initial_opinions=x_dummy,
        control_resistance=np.zeros(N, dtype=float),
    )

    A = np.asarray(graph_env.connectivity_matrix, dtype=float).copy()
    assert A.shape == (N, N)
    return A


def build_unstructured_x0(opinion_seed: int) -> np.ndarray:
    base = np.linspace(0.0, 1.0, N)
    rng = np.random.default_rng(920_000 + int(opinion_seed))
    return rng.permutation(base).astype(float)


UNSTRUCTURED_A_BY_SEED = {
    int(seed): build_unstructured_adjacency(int(seed))
    for seed in UNSTRUCTURED_TOPOLOGY_SEEDS
}

print("Built unstructured graph seeds:", sorted(UNSTRUCTURED_A_BY_SEED))

Built unstructured graph seeds: [3, 4, 5, 6, 7]


## Exact structured benchmark generators

In [5]:
HK_LOW_CLUSTER = tuple(range(0, 7))
HK_HIGH_CLUSTER = tuple(range(7, 15))
HK_LOW_HUB = 0
HK_HIGH_HUB = 7

BROADCAST_HUBS = (0, 1)
BROADCAST_LEAVES = tuple(range(2, N))
BROADCAST_GROUP_A = tuple(range(2, 8))
BROADCAST_GROUP_B = tuple(range(8, N))


def build_hk_graph(
    family_name: str,
    environment_seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    params = HK_FAMILIES[family_name]
    raw = np.zeros((N, N), dtype=float)

    for node in HK_LOW_CLUSTER:
        if node != HK_LOW_HUB:
            _add_undirected(
                raw,
                HK_LOW_HUB,
                node,
                params["hub_leaf_weight"],
            )

    for node in HK_HIGH_CLUSTER:
        if node != HK_HIGH_HUB:
            _add_undirected(
                raw,
                HK_HIGH_HUB,
                node,
                params["hub_leaf_weight"],
            )

    for cluster, hub in [
        (HK_LOW_CLUSTER, HK_LOW_HUB),
        (HK_HIGH_CLUSTER, HK_HIGH_HUB),
    ]:
        leaves = [node for node in cluster if node != hub]
        pattern = params["peer_pattern"]
        w = float(params["peer_weight"])

        if pattern == "ring":
            for left, right in zip(
                leaves,
                leaves[1:] + leaves[:1],
            ):
                _add_undirected(raw, left, right, w)

        elif pattern == "paired":
            for index in range(0, len(leaves) - 1, 2):
                _add_undirected(
                    raw,
                    leaves[index],
                    leaves[index + 1],
                    w,
                )
            if len(leaves) % 2 == 1:
                _add_undirected(
                    raw,
                    leaves[-1],
                    leaves[0],
                    0.5 * w,
                )

        elif pattern == "dense":
            for pos, left in enumerate(leaves):
                for right in leaves[pos + 1:]:
                    _add_undirected(raw, left, right, w)

        else:
            raise ValueError(pattern)

    # The sole inter-community topological edge.
    _add_undirected(
        raw,
        HK_LOW_HUB,
        HK_HIGH_HUB,
        params["bridge_weight"],
    )

    rng = np.random.default_rng(
        10_000
        + 100 * list(HK_FAMILIES).index(family_name)
        + int(environment_seed)
    )
    raw = _jitter_positive_edges(
        raw,
        rng,
        EDGE_WEIGHT_JITTER_FRAC,
    )
    return raw, _normalize_rows(raw)


def build_hk_x0(environment_seed: int) -> np.ndarray:
    base = np.array(
        [
            0.35,
            0.27, 0.28, 0.29, 0.30, 0.31, 0.32,
            0.64,
            0.65, 0.66, 0.67, 0.68, 0.69, 0.70, 0.71,
        ],
        dtype=float,
    )
    rng = np.random.default_rng(20_000 + int(environment_seed))
    return np.clip(
        base
        + rng.uniform(
            -HK_OPINION_JITTER,
            HK_OPINION_JITTER,
            size=N,
        ),
        0.0,
        1.0,
    )


def build_broadcaster_graph(
    family_name: str,
    environment_seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    params = BROADCAST_FAMILIES[family_name]
    h = float(params["hub_influence_total"])
    kind = params["kind"]

    raw = np.zeros((N, N), dtype=float)
    leaves = list(BROADCAST_LEAVES)

    for idx, leaf in enumerate(leaves):
        prev_leaf = leaves[(idx - 1) % len(leaves)]
        next_leaf = leaves[(idx + 1) % len(leaves)]

        if kind == "dual":
            raw[leaf, 0] = 0.50 * h
            raw[leaf, 1] = 0.50 * h

        elif kind == "split":
            primary = 0 if leaf in BROADCAST_GROUP_A else 1
            secondary = 1 if primary == 0 else 0
            raw[leaf, primary] = 0.85 * h
            raw[leaf, secondary] = 0.15 * h

        elif kind == "asymmetric":
            raw[leaf, 0] = 0.75 * h
            raw[leaf, 1] = 0.25 * h

        else:
            raise ValueError(kind)

        remaining = 1.0 - h
        raw[leaf, prev_leaf] += 0.50 * remaining
        raw[leaf, next_leaf] += 0.50 * remaining

    raw[0, 1] = 0.20
    raw[1, 0] = 0.20

    for leaf in BROADCAST_GROUP_A:
        raw[0, leaf] += 0.80 / len(BROADCAST_GROUP_A)
    for leaf in BROADCAST_GROUP_B:
        raw[1, leaf] += 0.80 / len(BROADCAST_GROUP_B)

    rng = np.random.default_rng(
        30_000
        + 100 * list(BROADCAST_FAMILIES).index(family_name)
        + int(environment_seed)
    )
    raw = _jitter_positive_edges(
        raw,
        rng,
        EDGE_WEIGHT_JITTER_FRAC,
    )
    return raw, _normalize_rows(raw)


def build_broadcast_x0(
    *,
    hub_level: float,
    leaf_level: float,
    environment_seed: int,
) -> np.ndarray:
    x0 = np.empty(N, dtype=float)

    x0[0] = float(hub_level) - 0.01
    x0[1] = float(hub_level) + 0.01

    offsets = np.linspace(
        -0.035,
        0.035,
        len(BROADCAST_LEAVES),
    )
    x0[list(BROADCAST_LEAVES)] = float(leaf_level) + offsets

    rng = np.random.default_rng(40_000 + int(environment_seed))
    x0 += rng.uniform(
        -BROADCAST_OPINION_JITTER,
        BROADCAST_OPINION_JITTER,
        size=N,
    )

    return np.clip(x0, 0.0, 1.0)

## Build the exact 195-environment registry

In [6]:
ENVIRONMENT_SPECS: list[dict[str, Any]] = []

# ---------------------------------------------------------------------------
# Unstructured: all six dynamics, same graph/opinion seed grid.
# ---------------------------------------------------------------------------
for dynamics, dynamics_params in UNSTRUCTURED_DYNAMICS.items():
    for topology_seed in UNSTRUCTURED_TOPOLOGY_SEEDS:
        for opinion_seed in UNSTRUCTURED_OPINION_SEEDS:
            A = np.asarray(
                UNSTRUCTURED_A_BY_SEED[int(topology_seed)],
                dtype=float,
            ).copy()
            x0 = build_unstructured_x0(int(opinion_seed))

            ENVIRONMENT_SPECS.append(
                {
                    "environment_index": len(ENVIRONMENT_SPECS),
                    "environment_id": (
                        f"unstructured__{dynamics}"
                        f"__topo{int(topology_seed):02d}"
                        f"__x{int(opinion_seed):02d}"
                    ),
                    "scenario_class": "unstructured_random",
                    "family": "barabasi_albert_spread",
                    "dynamics": dynamics,
                    "topology_seed": int(topology_seed),
                    "opinion_seed": int(opinion_seed),
                    "environment_seed": None,
                    "dynamics_params": dict(dynamics_params),
                    "A": A,
                    "x0": x0,
                    "v_true": centrality_from_A(A),
                }
            )

# ---------------------------------------------------------------------------
# Structured HK.
# ---------------------------------------------------------------------------
for family in HK_FAMILIES:
    for environment_seed in STRUCTURED_ENVIRONMENT_SEEDS:
        raw, A = build_hk_graph(family, environment_seed)
        x0 = build_hk_x0(environment_seed)

        ENVIRONMENT_SPECS.append(
            {
                "environment_index": len(ENVIRONMENT_SPECS),
                "environment_id": (
                    f"structured__hegselmannkrause"
                    f"__{family}__env{environment_seed:02d}"
                ),
                "scenario_class": "structured_mechanism",
                "family": family,
                "dynamics": "hegselmannkrause",
                "topology_seed": None,
                "opinion_seed": None,
                "environment_seed": int(environment_seed),
                "dynamics_params": {
                    "hk_epsilon": STRUCTURED_HK_EPSILON,
                    "hk_include_self": True,
                },
                "raw": raw,
                "A": A,
                "x0": x0,
                "v_true": centrality_from_A(A),
            }
        )

# ---------------------------------------------------------------------------
# Structured FJ / nonlinear influence.
# ---------------------------------------------------------------------------
for dynamics in ["friedkinjohnsen", "nonlinearinfluence"]:
    for family in BROADCAST_FAMILIES:
        for environment_seed in STRUCTURED_ENVIRONMENT_SEEDS:
            raw, A = build_broadcaster_graph(
                family,
                environment_seed,
            )

            if dynamics == "friedkinjohnsen":
                x0 = build_broadcast_x0(
                    hub_level=FJ_HUB_LEVEL,
                    leaf_level=FJ_LEAF_LEVEL,
                    environment_seed=environment_seed,
                )
                dynamics_params = {
                    "fj_lambda": STRUCTURED_FJ_LAMBDA,
                }
            else:
                x0 = build_broadcast_x0(
                    hub_level=NLI_HUB_LEVEL,
                    leaf_level=NLI_LEAF_LEVEL,
                    environment_seed=environment_seed,
                )
                dynamics_params = {
                    "nonlinear_beta": STRUCTURED_NLI_BETA,
                }

            ENVIRONMENT_SPECS.append(
                {
                    "environment_index": len(ENVIRONMENT_SPECS),
                    "environment_id": (
                        f"structured__{dynamics}"
                        f"__{family}__env{environment_seed:02d}"
                    ),
                    "scenario_class": "structured_mechanism",
                    "family": family,
                    "dynamics": dynamics,
                    "topology_seed": None,
                    "opinion_seed": None,
                    "environment_seed": int(environment_seed),
                    "dynamics_params": dynamics_params,
                    "raw": raw,
                    "A": A,
                    "x0": x0,
                    "v_true": centrality_from_A(A),
                }
            )

assert len(ENVIRONMENT_SPECS) == 195
assert len({spec["environment_id"] for spec in ENVIRONMENT_SPECS}) == 195

registry_rows = [
    {
        key: spec.get(key)
        for key in [
            "environment_index",
            "environment_id",
            "scenario_class",
            "family",
            "dynamics",
            "topology_seed",
            "opinion_seed",
            "environment_seed",
        ]
    }
    for spec in ENVIRONMENT_SPECS
]

registry_df = pd.DataFrame(registry_rows)

display(
    registry_df.groupby(
        ["scenario_class", "dynamics"],
        as_index=False,
    ).size()
)

print("Distinct environments:", len(ENVIRONMENT_SPECS))
print("Static baseline runs:", len(ENVIRONMENT_SPECS))
print("Analytic baseline runs:", sum(spec["dynamics"] in ANALYTIC_DYNAMICS for spec in ENVIRONMENT_SPECS))

,scenario_class,dynamics,size
0,structured_mechanism,friedkinjohnsen,15
1,structured_mechanism,hegselmannkrause,15
2,structured_mechanism,nonlinearinfluence,15
3,unstructured_random,coca,25
4,unstructured_random,friedkinjohnsen,25
5,unstructured_random,hegselmannkrause,25
6,unstructured_random,laplacian,25
7,unstructured_random,nonlinearinfluence,25
8,unstructured_random,repulsion,25


Distinct environments: 195
Static baseline runs: 195
Analytic baseline runs: 90


## Environment and reference-policy definitions

`true_graph_static` is the existing reference and uses the fixed centrality of
the exact \(A\).

`true_graph_analytic` uses the exact state-dependent interaction matrix
\(M^\star(x)\) at every controlled campaign. The resulting centrality is passed
to the **same** `centrality_based_continuous_control` routine.

The analytic matrix is deliberately **not row-normalized**. The learned
controller also forms \(\hat A\odot\alpha_\phi(x)\) without renormalizing the
rows, so normalization here would change the mathematical object being
compared.


In [7]:
def make_env(
    spec: dict[str, Any],
    *,
    seed: int,
) -> NetworkGraph:
    params = dict(spec["dynamics_params"])

    kwargs = dict(
        connectivity_matrix=np.asarray(
            spec["A"],
            dtype=float,
        ).copy(),
        num_agents=N,
        max_u=MAX_U,
        desired_opinion=TARGET,
        t_campaign=T_CAMPAIGN,
        t_s=T_S,
        initial_opinions=np.asarray(
            spec["x0"],
            dtype=float,
        ).copy(),
        control_resistance=np.zeros(N, dtype=float),
        dynamics_model=spec["dynamics"],
        budget=1000.0,
        max_steps=NUM_CAMPAIGNS + 20,
        opinion_end_tolerance=0.01,
        control_beta=0.4,
        normalize_reward=True,
        terminal_reward=0.0,
        terminate_when_converged=False,
        seed=int(seed),
    )

    if spec["dynamics"] == "hegselmannkrause":
        kwargs["hk_epsilon"] = float(
            params["hk_epsilon"]
        )
        kwargs["hk_include_self"] = bool(
            params["hk_include_self"]
        )

    elif spec["dynamics"] == "friedkinjohnsen":
        kwargs["fj_lambda"] = float(
            params["fj_lambda"]
        )
        kwargs["fj_prejudice"] = np.asarray(
            spec["x0"],
            dtype=float,
        ).copy()

    elif spec["dynamics"] == "nonlinearinfluence":
        kwargs["nonlinear_beta"] = float(
            params["nonlinear_beta"]
        )

    elif spec["dynamics"] == "repulsion":
        kwargs["repulsion_epsilon"] = float(
            params["repulsion_epsilon"]
        )
        kwargs["repulsion_strength"] = float(
            params["repulsion_strength"]
        )

    return NetworkGraph(**kwargs)


def set_state(
    env: NetworkGraph,
    x0: np.ndarray,
) -> None:
    env.reset()
    env.opinions = np.asarray(
        x0,
        dtype=float,
    ).copy()

    if hasattr(env, "state"):
        try:
            env.state = np.asarray(
                x0,
                dtype=float,
            ).copy()
        except Exception:
            pass


def analytic_control_matrix(
    spec: dict[str, Any],
    state: np.ndarray,
) -> np.ndarray:
    """
    Exact control-relevant interaction matrix for the dynamics for which
    the environment law admits the same nonnegative pairwise form used by
    the learned nonlinear controller.
    """
    A = np.asarray(spec["A"], dtype=float)
    x = np.asarray(state, dtype=float).reshape(-1)
    dynamics = spec["dynamics"]

    if dynamics == "laplacian":
        M = A.copy()

    elif dynamics == "coca":
        susceptibility = x * (1.0 - x)
        M = susceptibility[:, None] * A

    elif dynamics == "nonlinearinfluence":
        beta = float(
            spec["dynamics_params"]["nonlinear_beta"]
        )
        diff = x[None, :] - x[:, None]

        alpha = np.empty_like(diff)
        near_zero = np.abs(diff) <= 1e-12

        alpha[near_zero] = beta
        alpha[~near_zero] = (
            np.tanh(beta * diff[~near_zero])
            / diff[~near_zero]
        )

        M = A * alpha

    else:
        raise ValueError(
            f"No analytic centrality matrix is defined for {dynamics!r}."
        )

    M = np.asarray(M, dtype=float)
    np.fill_diagonal(M, 0.0)

    if not np.all(np.isfinite(M)):
        raise RuntimeError(
            f"{spec['environment_id']}: non-finite analytic matrix."
        )

    if np.min(M) < -1e-12:
        raise RuntimeError(
            f"{spec['environment_id']}: analytic matrix became signed."
        )

    M = np.maximum(M, 0.0)
    return M


def laplacian_nullity(
    matrix: np.ndarray,
) -> int:
    """
    Numerical nullity of the matrix Laplacian.

    Nullity > 1 means the global centrality vector is not mathematically
    unique. We record this instead of silently regularizing the graph.
    """
    L = np.asarray(
        compute_laplacian(
            np.asarray(matrix, dtype=float)
        ),
        dtype=float,
    )

    singular = np.linalg.svd(
        L,
        compute_uv=False,
    )

    scale = max(
        1.0,
        float(singular[0]) if singular.size else 1.0,
    )
    tol = (
        max(L.shape)
        * np.finfo(float).eps
        * scale
        * 100.0
    )

    rank = int(np.sum(singular > tol))
    return int(L.shape[0] - rank)


def control_from_centrality(
    env: NetworkGraph,
    v: np.ndarray,
) -> np.ndarray:
    action, _ = centrality_based_continuous_control(
        env,
        float(B_CAMPAIGN),
        v=np.asarray(v, dtype=float),
    )
    return np.asarray(action, dtype=float)


def run_reference_policy(
    spec: dict[str, Any],
    *,
    policy: str,
) -> dict[str, Any]:
    if (
        policy == "true_graph_analytic"
        and spec["dynamics"] not in ANALYTIC_DYNAMICS
    ):
        raise ValueError(
            f"Analytic reference not defined for {spec['dynamics']}."
        )

    env = make_env(
        spec,
        seed=500_000 + int(spec["environment_index"]),
    )
    set_state(env, spec["x0"])

    states = [
        np.asarray(
            spec["x0"],
            dtype=float,
        ).copy()
    ]
    actions = []
    rewards = []
    intermediate_states_list = []

    control_matrices = []
    centralities = []
    nullities = []

    for campaign in range(NUM_CAMPAIGNS):
        state_pre = np.asarray(
            env.opinions,
            dtype=float,
        ).copy()

        if policy == "true_graph_static":
            M = np.asarray(
                spec["A"],
                dtype=float,
            ).copy()
            v = np.asarray(
                spec["v_true"],
                dtype=float,
            ).copy()

        elif policy == "true_graph_analytic":
            M = analytic_control_matrix(
                spec,
                state_pre,
            )
            v = centrality_from_A(M)

        else:
            raise ValueError(policy)

        control_matrices.append(M)
        centralities.append(v)
        nullities.append(
            laplacian_nullity(M)
        )

        if campaign == 0:
            action = np.zeros(
                N,
                dtype=float,
            )
        else:
            action = control_from_centrality(
                env,
                v,
            )

        x_next, reward, done, trunc, info = env.step(
            action
        )

        states.append(
            np.asarray(
                x_next,
                dtype=float,
            ).copy()
        )
        actions.append(
            np.asarray(
                action,
                dtype=float,
            ).copy()
        )
        rewards.append(float(reward))
        intermediate_states_list.append(
            np.asarray(
                info["intermediate_states"],
                dtype=float,
            ).copy()
        )

        if done or trunc:
            break

    return {
        "policy": policy,
        "states": np.asarray(
            states,
            dtype=float,
        ),
        "actions": np.asarray(
            actions,
            dtype=float,
        ),
        "rewards": np.asarray(
            rewards,
            dtype=float,
        ),
        "intermediate_states_list": (
            intermediate_states_list
        ),
        "control_matrices": np.asarray(
            control_matrices,
            dtype=float,
        ),
        "centralities": np.asarray(
            centralities,
            dtype=float,
        ),
        "laplacian_nullities": np.asarray(
            nullities,
            dtype=int,
        ),
    }


## Summary metrics and persistent cache

In [8]:
def _json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {
            str(k): _json_safe(v)
            for k, v in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [_json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        x = float(value)
        return x if np.isfinite(x) else None
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, float):
        return value if np.isfinite(value) else None
    return value


def sha256_json(
    payload: dict[str, Any],
) -> str:
    return hashlib.sha256(
        json.dumps(
            _json_safe(payload),
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()


SCIENTIFIC_CONFIG = {
    "pipeline_version": PIPELINE_VERSION,
    "N": N,
    "target": TARGET,
    "num_campaigns": NUM_CAMPAIGNS,
    "t_campaign": T_CAMPAIGN,
    "t_s": T_S,
    "max_u": MAX_U,
    "total_controlled_budget": TOTAL_CONTROLLED_BUDGET,
    "B_campaign": B_CAMPAIGN,
    "unstructured_topology_seeds": (
        UNSTRUCTURED_TOPOLOGY_SEEDS
    ),
    "unstructured_opinion_seeds": (
        UNSTRUCTURED_OPINION_SEEDS
    ),
    "unstructured_dynamics": (
        UNSTRUCTURED_DYNAMICS
    ),
    "structured_environment_seeds": (
        STRUCTURED_ENVIRONMENT_SEEDS
    ),
    "hk_families": HK_FAMILIES,
    "broadcast_families": BROADCAST_FAMILIES,
    "structured_hk_epsilon": (
        STRUCTURED_HK_EPSILON
    ),
    "structured_fj_lambda": (
        STRUCTURED_FJ_LAMBDA
    ),
    "structured_nli_beta": (
        STRUCTURED_NLI_BETA
    ),
    "analytic_dynamics": sorted(
        ANALYTIC_DYNAMICS
    ),
    "analytic_matrix_definition": {
        "laplacian": "A",
        "coca": "diag(x*(1-x)) @ A",
        "nonlinearinfluence": (
            "A_ij * tanh(beta*(x_j-x_i))/(x_j-x_i), "
            "limit beta at zero"
        ),
    },
}

CONFIG_HASH = sha256_json(
    SCIENTIFIC_CONFIG
)

IMPLEMENTATION_SOURCES = {
    "NetworkGraph": (
        str(
            inspect.getsourcefile(
                NetworkGraph
            )
        )
    ),
    "centrality_based_continuous_control": (
        str(
            inspect.getsourcefile(
                centrality_based_continuous_control
            )
        )
    ),
}

print("Config hash:", CONFIG_HASH[:16])


def policy_cache_dir(
    spec: dict[str, Any],
    policy: str,
) -> Path:
    payload = {
        "environment_id": spec["environment_id"],
        "policy": policy,
        "config_hash": CONFIG_HASH,
    }
    key = sha256_json(payload)

    return (
        CACHE_ROOT
        / spec["environment_id"]
        / f"{policy}__{key[:16]}"
    )


def rollout_metrics(
    rollout: dict[str, Any],
) -> dict[str, Any]:
    states = np.asarray(
        rollout["states"],
        dtype=float,
    )
    final = states[-1]
    campaign_means = states.mean(axis=1)

    nullities = np.asarray(
        rollout["laplacian_nullities"],
        dtype=int,
    )

    return {
        "final_mean": float(
            final.mean()
        ),
        "mean_over_campaigns": float(
            campaign_means.mean()
        ),
        "final_min": float(
            final.min()
        ),
        "final_max": float(
            final.max()
        ),
        "final_mean_distance_to_target": float(
            np.mean(
                np.abs(
                    TARGET - final
                )
            )
        ),
        "max_laplacian_nullity": int(
            nullities.max()
        ),
        "fraction_campaigns_nonunique_centrality": float(
            np.mean(
                nullities > 1
            )
        ),
    }


def save_rollout_npz(
    path: Path,
    rollout: dict[str, Any],
) -> None:
    arrays: dict[str, np.ndarray] = {
        "states": np.asarray(
            rollout["states"],
            dtype=float,
        ),
        "actions": np.asarray(
            rollout["actions"],
            dtype=float,
        ),
        "rewards": np.asarray(
            rollout["rewards"],
            dtype=float,
        ),
        "control_matrices": np.asarray(
            rollout["control_matrices"],
            dtype=float,
        ),
        "centralities": np.asarray(
            rollout["centralities"],
            dtype=float,
        ),
        "laplacian_nullities": np.asarray(
            rollout["laplacian_nullities"],
            dtype=int,
        ),
    }

    for campaign, inter in enumerate(
        rollout["intermediate_states_list"]
    ):
        arrays[
            f"intermediate__{campaign:02d}"
        ] = np.asarray(
            inter,
            dtype=float,
        )

    tmp = path.with_suffix(".tmp")
    with tmp.open("wb") as handle:
        np.savez_compressed(
            handle,
            **arrays,
        )
    os.replace(
        tmp,
        path,
    )


def cache_complete(
    spec: dict[str, Any],
    policy: str,
) -> bool:
    directory = policy_cache_dir(
        spec,
        policy,
    )

    manifest_path = directory / "manifest.json"
    summary_path = directory / "summary.json"
    rollout_path = directory / "rollout.npz"

    if not all(
        p.exists()
        for p in [
            manifest_path,
            summary_path,
            rollout_path,
        ]
    ):
        return False

    try:
        manifest = json.loads(
            manifest_path.read_text(
                encoding="utf-8"
            )
        )
    except Exception:
        return False

    return (
        manifest.get("status") == "success"
        and manifest.get("config_hash") == CONFIG_HASH
        and manifest.get("environment_id")
        == spec["environment_id"]
        and manifest.get("policy") == policy
    )


def ensure_policy(
    spec: dict[str, Any],
    policy: str,
) -> dict[str, Any]:
    directory = policy_cache_dir(
        spec,
        policy,
    )

    if cache_complete(
        spec,
        policy,
    ):
        return json.loads(
            (
                directory / "summary.json"
            ).read_text(
                encoding="utf-8"
            )
        )

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    rollout = run_reference_policy(
        spec,
        policy=policy,
    )

    row = {
        "environment_id": (
            spec["environment_id"]
        ),
        "scenario_class": (
            spec["scenario_class"]
        ),
        "family": spec["family"],
        "dynamics": spec["dynamics"],
        "topology_seed": (
            spec.get("topology_seed")
        ),
        "opinion_seed": (
            spec.get("opinion_seed")
        ),
        "environment_seed": (
            spec.get("environment_seed")
        ),
        "policy": policy,
        **rollout_metrics(rollout),
    }

    save_rollout_npz(
        directory / "rollout.npz",
        rollout,
    )

    (
        directory / "summary.json"
    ).write_text(
        json.dumps(
            _json_safe(row),
            indent=2,
        ),
        encoding="utf-8",
    )

    (
        directory / "manifest.json"
    ).write_text(
        json.dumps(
            {
                "status": "success",
                "study_name": STUDY_NAME,
                "pipeline_version": PIPELINE_VERSION,
                "config_hash": CONFIG_HASH,
                "git_commit": GIT_COMMIT,
                "environment_id": (
                    spec["environment_id"]
                ),
                "policy": policy,
                "completed_at_utc": (
                    datetime.now(
                        timezone.utc
                    ).isoformat()
                ),
            },
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )

    return row


Config hash: 13dcc30e0fa72103


## Run the deterministic references

This is intentionally a single direct notebook run. There is no SGD, no neural
network fitting, and no learning-seed repetition.

Expected work:

- 195 static true-graph rollouts;
- 90 analytic true-model rollouts;
- 285 deterministic rollouts total.

Completed policy/environment pairs are cached individually and reused on rerun.


In [9]:
rows = []

for position, spec in enumerate(
    ENVIRONMENT_SPECS,
    start=1,
):
    print(
        f"[{position:03d}/{len(ENVIRONMENT_SPECS)}] "
        f"{spec['environment_id']}"
    )

    rows.append(
        ensure_policy(
            spec,
            "true_graph_static",
        )
    )

    if (
        spec["dynamics"]
        in ANALYTIC_DYNAMICS
    ):
        rows.append(
            ensure_policy(
                spec,
                "true_graph_analytic",
            )
        )

results_df = pd.DataFrame(rows)

print("\nRows:", len(results_df))
display(
    results_df.groupby(
        [
            "policy",
            "scenario_class",
            "dynamics",
        ],
        as_index=False,
    ).size()
)


[001/195] unstructured__laplacian__topo03__x00
[002/195] unstructured__laplacian__topo03__x01
[003/195] unstructured__laplacian__topo03__x02
[004/195] unstructured__laplacian__topo03__x04
[005/195] unstructured__laplacian__topo03__x05
[006/195] unstructured__laplacian__topo04__x00
[007/195] unstructured__laplacian__topo04__x01
[008/195] unstructured__laplacian__topo04__x02
[009/195] unstructured__laplacian__topo04__x04
[010/195] unstructured__laplacian__topo04__x05
[011/195] unstructured__laplacian__topo05__x00
[012/195] unstructured__laplacian__topo05__x01
[013/195] unstructured__laplacian__topo05__x02
[014/195] unstructured__laplacian__topo05__x04
[015/195] unstructured__laplacian__topo05__x05
[016/195] unstructured__laplacian__topo06__x00
[017/195] unstructured__laplacian__topo06__x01
[018/195] unstructured__laplacian__topo06__x02
[019/195] unstructured__laplacian__topo06__x04
[020/195] unstructured__laplacian__topo06__x05
[021/195] unstructured__laplacian__topo07__x00
[022/195] uns

,policy,scenario_class,dynamics,size
0,true_graph_analytic,structured_mechanism,nonlinearinfluence,15
1,true_graph_analytic,unstructured_random,coca,25
2,true_graph_analytic,unstructured_random,laplacian,25
3,true_graph_analytic,unstructured_random,nonlinearinfluence,25
4,true_graph_static,structured_mechanism,friedkinjohnsen,15
5,true_graph_static,structured_mechanism,hegselmannkrause,15
6,true_graph_static,structured_mechanism,nonlinearinfluence,15
7,true_graph_static,unstructured_random,coca,25
8,true_graph_static,unstructured_random,friedkinjohnsen,25
9,true_graph_static,unstructured_random,hegselmannkrause,25


## Strict completion checks and sanity diagnostics

In [10]:
static_df = results_df[
    results_df["policy"]
    == "true_graph_static"
].copy()

analytic_df = results_df[
    results_df["policy"]
    == "true_graph_analytic"
].copy()

assert len(static_df) == 195
assert static_df[
    "environment_id"
].nunique() == 195

assert len(analytic_df) == 90
assert set(
    analytic_df["dynamics"].unique()
) == ANALYTIC_DYNAMICS

expected_analytic_counts = {
    "laplacian": 25,
    "coca": 25,
    "nonlinearinfluence": 40,
}

actual_analytic_counts = (
    analytic_df["dynamics"]
    .value_counts()
    .to_dict()
)

assert (
    actual_analytic_counts
    == expected_analytic_counts
), actual_analytic_counts

# Linear analytic and static references must be exactly the same control
# construction. Check both summary values and stored actions/states.
linear_static = static_df[
    static_df["dynamics"]
    == "laplacian"
][
    [
        "environment_id",
        "final_mean",
        "mean_over_campaigns",
    ]
].rename(
    columns={
        "final_mean": "static_final",
        "mean_over_campaigns": (
            "static_path"
        ),
    }
)

linear_analytic = analytic_df[
    analytic_df["dynamics"]
    == "laplacian"
][
    [
        "environment_id",
        "final_mean",
        "mean_over_campaigns",
    ]
].rename(
    columns={
        "final_mean": "analytic_final",
        "mean_over_campaigns": (
            "analytic_path"
        ),
    }
)

linear_check = linear_static.merge(
    linear_analytic,
    on="environment_id",
    validate="one_to_one",
)

assert np.allclose(
    linear_check["static_final"],
    linear_check["analytic_final"],
    atol=1e-12,
    rtol=0.0,
)
assert np.allclose(
    linear_check["static_path"],
    linear_check["analytic_path"],
    atol=1e-12,
    rtol=0.0,
)

# Stronger artifact-level equality for linear.
for spec in ENVIRONMENT_SPECS:
    if spec["dynamics"] != "laplacian":
        continue

    static_arrays = np.load(
        policy_cache_dir(
            spec,
            "true_graph_static",
        )
        / "rollout.npz",
        allow_pickle=False,
    )
    analytic_arrays = np.load(
        policy_cache_dir(
            spec,
            "true_graph_analytic",
        )
        / "rollout.npz",
        allow_pickle=False,
    )

    assert np.allclose(
        static_arrays["actions"],
        analytic_arrays["actions"],
        atol=1e-12,
        rtol=0.0,
    )
    assert np.allclose(
        static_arrays["states"],
        analytic_arrays["states"],
        atol=1e-12,
        rtol=0.0,
    )

print("LINEAR SANITY CHECK PASSED: analytic == static exactly.")

# COCA may have more than one zero-Laplacian mode at states with multiple
# zero-susceptibility agents. Record this transparently rather than adding
# an arbitrary epsilon to the exact dynamics.
nullity_summary = (
    analytic_df.groupby(
        [
            "scenario_class",
            "dynamics",
        ],
        as_index=False,
    )
    .agg(
        n_environments=(
            "environment_id",
            "size",
        ),
        max_nullity=(
            "max_laplacian_nullity",
            "max",
        ),
        mean_fraction_nonunique=(
            "fraction_campaigns_nonunique_centrality",
            "mean",
        ),
        max_fraction_nonunique=(
            "fraction_campaigns_nonunique_centrality",
            "max",
        ),
    )
)

print("\nAnalytic-centrality uniqueness diagnostics:")
display(
    nullity_summary.round(6)
)

results_df.to_csv(
    RESULTS_DIR
    / "baseline_summary_all.csv",
    index=False,
)

static_df.to_csv(
    RESULTS_DIR
    / "true_graph_static_all_195.csv",
    index=False,
)

analytic_df.to_csv(
    RESULTS_DIR
    / "true_graph_analytic_supported_90.csv",
    index=False,
)

nullity_summary.to_csv(
    RESULTS_DIR
    / "analytic_centrality_uniqueness_diagnostics.csv",
    index=False,
)


LINEAR SANITY CHECK PASSED: analytic == static exactly.

Analytic-centrality uniqueness diagnostics:


,scenario_class,dynamics,n_environments,max_nullity,mean_fraction_nonunique,max_fraction_nonunique
0,structured_mechanism,nonlinearinfluence,15,1,0.000,0.0
1,unstructured_random,coca,25,2,0.676,1.0
2,unstructured_random,laplacian,25,1,0.000,0.0
3,unstructured_random,nonlinearinfluence,25,1,0.000,0.0


## Quick outcome table

In [11]:
static_supported = static_df[
    static_df["dynamics"].isin(
        ANALYTIC_DYNAMICS
    )
][
    [
        "environment_id",
        "scenario_class",
        "family",
        "dynamics",
        "final_mean",
        "mean_over_campaigns",
    ]
].rename(
    columns={
        "final_mean": "static_final",
        "mean_over_campaigns": (
            "static_path"
        ),
    }
)

analytic_compare = analytic_df[
    [
        "environment_id",
        "final_mean",
        "mean_over_campaigns",
        "max_laplacian_nullity",
        "fraction_campaigns_nonunique_centrality",
    ]
].rename(
    columns={
        "final_mean": "analytic_final",
        "mean_over_campaigns": (
            "analytic_path"
        ),
    }
)

comparison = static_supported.merge(
    analytic_compare,
    on="environment_id",
    validate="one_to_one",
)

comparison[
    "analytic_minus_static_final"
] = (
    comparison["analytic_final"]
    - comparison["static_final"]
)

comparison[
    "analytic_minus_static_path"
] = (
    comparison["analytic_path"]
    - comparison["static_path"]
)

quick_summary = (
    comparison.groupby(
        [
            "scenario_class",
            "dynamics",
        ],
        as_index=False,
    )
    .agg(
        n_environments=(
            "environment_id",
            "size",
        ),
        static_final=(
            "static_final",
            "mean",
        ),
        analytic_final=(
            "analytic_final",
            "mean",
        ),
        analytic_minus_static_final=(
            "analytic_minus_static_final",
            "mean",
        ),
        static_path=(
            "static_path",
            "mean",
        ),
        analytic_path=(
            "analytic_path",
            "mean",
        ),
        analytic_minus_static_path=(
            "analytic_minus_static_path",
            "mean",
        ),
        max_nullity=(
            "max_laplacian_nullity",
            "max",
        ),
        mean_fraction_nonunique=(
            "fraction_campaigns_nonunique_centrality",
            "mean",
        ),
    )
)

display(
    quick_summary.round(6)
)

comparison.to_csv(
    RESULTS_DIR
    / "analytic_vs_static_environment_level.csv",
    index=False,
)
quick_summary.to_csv(
    RESULTS_DIR
    / "analytic_vs_static_quick_summary.csv",
    index=False,
)


,scenario_class,dynamics,n_environments,static_final,analytic_final,analytic_minus_static_final,static_path,analytic_path,analytic_minus_static_path,max_nullity,mean_fraction_nonunique
0,structured_mechanism,nonlinearinfluence,15,0.865575,0.865575,0.000000,0.641443,0.641443,0.000000,1,0.000
1,unstructured_random,coca,25,0.781852,0.643016,-0.138836,0.651869,0.585177,-0.066692,2,0.676
2,unstructured_random,laplacian,25,0.816446,0.816446,0.000000,0.683607,0.683607,0.000000,1,0.000
3,unstructured_random,nonlinearinfluence,25,0.844726,0.844771,0.000044,0.707532,0.707594,0.000062,1,0.000


## Completion marker

In [12]:
completion = {
    "status": "success",
    "study_name": STUDY_NAME,
    "pipeline_version": PIPELINE_VERSION,
    "git_commit": GIT_COMMIT,
    "config_hash": CONFIG_HASH,
    "n_environments": 195,
    "n_static_rollouts": 195,
    "n_analytic_rollouts": 90,
    "n_total_rollouts": 285,
    "analytic_dynamics": sorted(
        ANALYTIC_DYNAMICS
    ),
    "analytic_counts": (
        expected_analytic_counts
    ),
    "completed_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
}

(
    RESULTS_DIR
    / "COMPUTATION_COMPLETE.json"
).write_text(
    json.dumps(
        completion,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

print("=" * 88)
print("TRUE-GRAPH BASELINE COMPUTATION COMPLETE")
print("=" * 88)
print("Static references:", 195)
print("Analytic references:", 90)
print("Total rollouts:", 285)
print("Results:", RESULTS_DIR)
print()
print(
    "The true-A + learned-alpha ablation is intentionally "
    "not part of this final baseline experiment."
)


TRUE-GRAPH BASELINE COMPUTATION COMPLETE
Static references: 195
Analytic references: 90
Total rollouts: 285
Results: d:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\results\experiment_2026_09_28_true_graph_static_and_analytic_baselines

The true-A + learned-alpha ablation is intentionally not part of this final baseline experiment.
